In [1]:
!git clone https://github.com/Saanvi521/lung_fusion_agent.git
%cd lung_fusion_agent
!pip install -r requirements.txt -q

Cloning into 'lung_fusion_agent'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 78 (delta 39), reused 64 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 25.22 KiB | 8.41 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/lung_fusion_agent


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/hf_cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.environ["HF_HUB_DISABLE_XET"] = "1"

Mounted at /content/drive


In [3]:
from huggingface_hub import login
login()

In [4]:
from huggingface_hub import hf_hub_download

csv_path = hf_hub_download("kmmuleelab/Lung_Pathology_Image_JPG", "CLWD.csv", repo_type="dataset")
print(csv_path)

# image_dir = snapshot_download("kmmuleelab/Lung_Pathology_Image_JPG", repo_type="dataset", allow_patterns="*.jpg")
# print(image_dir)

/content/drive/MyDrive/hf_cache/hub/datasets--kmmuleelab--Lung_Pathology_Image_JPG/snapshots/973975ae466f8f7751f5a3032c1a0ffea62b1fa0/CLWD.csv


In [5]:
from src.data.tiling import tile_slide
from src.embeddings.uni2h import UNI2HEncoder
print("imports OK")

imports OK


In [6]:
!python -m src.data.make_split --csv {csv_path} --seed 42

Wrote artifacts/splits/split_seed42.csv
split
train    146
test      42
val       21
split           test  train  val
subtype                         
Acinar             7     23    3
Cribriform         3     11    1
In situ            9     31    5
Lepidic            4     15    2
Micropapillary     5     18    3
Papillary          8     28    4
Solid              6     20    3
Label-conflict patients: 5
Selection-bias check: {'n_multi_slide_patients': 146, 'n_selected_matches_majority': 143, 'agreement_rate': 0.979}


In [7]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [8]:
import pandas as pd
split = pd.read_csv("artifacts/splits/split_seed42.csv")
split.head(2).to_csv("artifacts/splits/split_TEST2.csv", index=False)

In [ ]:
# Cell 9 -- Stage A on just those 2 patients
!python -m scripts.02a_tile_slides \
  --split artifacts/splits/split_TEST2.csv \
  --out-dir /content/lung_fusion_test \
  --repo-id kmmuleelab/Lung_Pathology_Image_JPG

In [ ]:
# Cell 10 -- Stage B on just those 2 patients (uni2h only, just proving it works)
!python -m scripts.02b_extract_embeddings --model uni2h \
  --split artifacts/splits/split_TEST2.csv \
  --out-dir /content/lung_fusion_test

In [ ]:
# Cell 11 -- confirm shape
import numpy as np
emb = np.load(f"/content/lung_fusion_test/embeddings/uni2h/{split.iloc[0]['patient_id']}.npy")
print(emb.shape)  # should be (1536,)

In [10]:
# --- Cell 12: Stage A — tile every slide ONCE, cache to Drive (no model loaded) ---
!python -m scripts.02a_tile_slides \
  --split artifacts/splits/split_seed42.csv \
  --out-dir /content/drive/MyDrive/lung_fusion_data \
  --repo-id kmmuleelab/Lung_Pathology_Image_JPG

Listing repo files to resolve WSI_ID -> actual path (one-time)...
Found 386 jpgs in repo.
Loading all 3 models (one-time; ~10GB total, should fit a T4 comfortably now that image decoding no longer spikes memory)...
config.json: 100% 742/742 [00:00<00:00, 1.51MB/s]
model.safetensors: 100% 2.53G/2.53G [00:40<00:00, 62.2MB/s]
config.json: 100% 993/993 [00:00<00:00, 1.48MB/s]
configuration_genbio_pathfm.py: 100% 2.22k/2.22k [00:00<00:00, 471kB/s]
[transformers] A new version of the following files was downloaded from https://huggingface.co/genbio-ai/genbio-pathfm:
- configuration_genbio_pathfm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
modeling_genbio_pathfm.py: 100% 25.6k/25.6k [00:00<00:00, 8.51MB/s]
[transformers] A new version of the following files was downloaded from https://huggingface.co/genbio-ai/genbio-pathfm:
- modeling_genbio_pathfm.py
. Make sure to double-check they d

In [ ]:
# --- Cell 9: Stage B, model 1/3 ---
!python -m scripts.02b_extract_embeddings --model uni2h \
  --split artifacts/splits/split_seed42.csv \
  --out-dir /content/drive/MyDrive/lung_fusion_data

In [ ]:
# --- Cell 10: Stage B, model 2/3 ---
!python -m scripts.02b_extract_embeddings --model virchow2 \
  --split artifacts/splits/split_seed42.csv \
  --out-dir /content/drive/MyDrive/lung_fusion_data

In [ ]:
# --- Cell 11: Stage B, model 3/3 ---
!python -m scripts.02b_extract_embeddings --model genbio_pathfm \
  --split artifacts/splits/split_seed42.csv \
  --out-dir /content/drive/MyDrive/lung_fusion_data

In [ ]:
# --- Cell 12: sanity check ---
import json, os
stats = json.load(open("/content/drive/MyDrive/lung_fusion_data/embeddings/extraction_stats.json"))
print(json.dumps(stats, indent=2))
for model in ["uni2h", "virchow2", "genbio_pathfm"]:
    n_files = len(os.listdir(f"/content/drive/MyDrive/lung_fusion_data/embeddings/{model}"))
    print(model, n_files, "files")

In [ ]:
# --- Cell 13: Step 3 — run the fusion agent ---
!python -m scripts.03_run_agent \
  --split artifacts/splits/split_seed42.csv \
  --embeddings-dir /content/drive/MyDrive/lung_fusion_data/embeddings \
  --out-dir /content/drive/MyDrive/lung_fusion_data/agent_run \
  --max-trials 35

In [ ]:
# --- Cell 14: commented, run manually when you want to push ---
# !git config --global user.email "saanvi7168l@outlook.com"
# !git config --global user.name "Saanvi521"
# !git add artifacts/splits/
# !git commit -m "add real split from CLWD.csv"
# import getpass
# token = getpass.getpass("Paste your GitHub token:")
# !git push https://{token}@github.com/Saanvi521/lung_fusion_agent.git main